In [72]:
import pandas as pd
import numpy as np
import itertools

In [73]:
df = pd.read_excel('/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/tests_7-04/SW_fully_impregnated.xlsx')
df.head(10)

,Product,Min weight Homogenous material in Product,Max weight Homogenous material in Product,Min % Homogenous material in Product,Max % Homogenous material in Product,Homogenous Material,Min weight Tier 1 material in Homogenous material,Max weight Tier 1 material in Homogenous material,Min % Tier 1 material in Homogenous material,Max % Tier 1 material in Homogenous material,...,Tier 4 Supplier,Is alternative of tier 4 material,Is alternative of tier 4 material (of what?),Coupled to Tier 4 material (only present if coupled material is present),Going to another TRL?,CAS Tier 4,Documets.1,Notes.1,max % in homogenous material,CAS Total
0,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.675964e-07,No CAS
1,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.675964e-07,494793-67-8
2,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.669894e-08,494793-67-8
3,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.669894e-08,29911-28-2
4,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,No CAS
5,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,9.999991e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.999991e-01,No CAS
6,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,No CAS
7,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,3.000000e-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000e-08,No CAS


In [74]:
def clean_data(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    # normalizing yes/no to be case sensitive
    mapping = {
        "yes": "yes",
        "no": "no",
        "Yes": "yes",
        "No": "no"
    }
    df = df.apply(lambda col: col.map(mapping).fillna(col) if col.dtype == "object" else col)
    # add a row with an ID for each material
    df["row_id"] = range(1, len(df) + 1)
    return df

In [75]:
df = clean_data(df)
df.head()

,Product,Min weight Homogenous material in Product,Max weight Homogenous material in Product,Min % Homogenous material in Product,Max % Homogenous material in Product,Homogenous Material,Min weight Tier 1 material in Homogenous material,Max weight Tier 1 material in Homogenous material,Min % Tier 1 material in Homogenous material,Max % Tier 1 material in Homogenous material,...,Is alternative of tier 4 material,Is alternative of tier 4 material (of what?),Coupled to Tier 4 material (only present if coupled material is present),Going to another TRL?,CAS Tier 4,Documets.1,Notes.1,max % in homogenous material,CAS Total,row_id
0,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.675964e-07,No CAS,1
1,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.675964e-07,494793-67-8,2
2,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.669894e-08,494793-67-8,3
3,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.669894e-08,29911-28-2,4
4,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,No CAS,5


In [76]:
def get_final_material(row, tier_level=5):
    for i in range(tier_level, 0, -1):
        col = f"Tier {i} Material"
        if pd.notna(row.get(col)):
            return row[col]
    return None

def get_final_supplier(row, tier_level=5):
    for i in range(tier_level, 0, -1):
        col = f"Tier {i} Supplier"
        if pd.notna(row.get(col)):
            return row[col]
    return None

def get_final_CAS(row, tier_level=5):
    for i in range(tier_level, 0, -1):
        col = f"CAS Tier {i}"
        if pd.notna(row.get(col)):
            return row[col]
    return "No CAS"

def get_tier_depth(row, field="Material", tier_level=5):
    for i in range(tier_level, 0, -1):
        col = f"Tier {i} {field}"
        if pd.notna(row.get(col)):
            return i
    return None

def add_helper_columns(df):
    df = df.copy()
    df["CAS"] = df.apply(get_final_CAS, axis=1)
    df["final_material"] = df.apply(get_final_material, axis=1)
    df["final_supplier"] = df.apply(get_final_supplier, axis=1)
    df["tier_depth"] = df.apply(get_tier_depth, axis=1)
    return df

In [77]:
df = add_helper_columns(df)
df.head(10)

,Product,Min weight Homogenous material in Product,Max weight Homogenous material in Product,Min % Homogenous material in Product,Max % Homogenous material in Product,Homogenous Material,Min weight Tier 1 material in Homogenous material,Max weight Tier 1 material in Homogenous material,Min % Tier 1 material in Homogenous material,Max % Tier 1 material in Homogenous material,...,CAS Tier 4,Documets.1,Notes.1,max % in homogenous material,CAS Total,row_id,CAS,final_material,final_supplier,tier_depth
0,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,8.675964e-07,No CAS,1,No CAS,Areosol DPNB 50%/IPBC50%,Bodotex A/S,2
1,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,8.675964e-07,494793-67-8,2,494793-67-8,Penflufen,Bodotex A/S,2
2,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,2.669894e-08,494793-67-8,3,494793-67-8,Penflufen,Bodotex A/S,2
3,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,2.669894e-08,29911-28-2,4,29911-28-2,Areosol DPNB,Bodotex A/S,2
4,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,NaN,1.000000e+00,No CAS,5,No CAS,Wood Versowood,Versowood,1
5,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,9.999991e-01,...,NaN,NaN,NaN,9.999991e-01,No CAS,6,No CAS,Wood Er-Saha Oy,Er-Saha Oy,1
6,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,NaN,1.000000e+00,No CAS,7,No CAS,Wood Koskisen Oyj,Koskisen Oyj,1
7,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,3.000000e-08,...,NaN,NaN,NaN,3.000000e-08,No CAS,8,No CAS,Impregnation (SC200)\n,Bodotex A/S,1


In [78]:
def build_location(row, tier_level=5):
    path = [row.get('Product'), row.get('Homogenous Material')]

    for i in range(1, tier_level + 1):
        col1 = f"Tier {i} Material"
        col_2 = f"Tier {i} Supplier"
        val1 = row.get(col1)
        val2 = row.get(col_2)
        val = f"{val1} ({val2})"

        if pd.notna(val):
            path.append(str(val))

        # stop once we reach the final tier depth
        if i == row.get("tier_depth"):
            break

    return " → ".join(path) if path else None


In [79]:
def add_final_map(df):
    df = df.copy()
    df["final_material_map"] = df.apply(lambda r: build_location(r, tier_level=5), axis=1)
    return df
df = add_final_map(df)
df.head(10)

,Product,Min weight Homogenous material in Product,Max weight Homogenous material in Product,Min % Homogenous material in Product,Max % Homogenous material in Product,Homogenous Material,Min weight Tier 1 material in Homogenous material,Max weight Tier 1 material in Homogenous material,Min % Tier 1 material in Homogenous material,Max % Tier 1 material in Homogenous material,...,Documets.1,Notes.1,max % in homogenous material,CAS Total,row_id,CAS,final_material,final_supplier,tier_depth,final_material_map
0,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,8.675964e-07,No CAS,1,No CAS,Areosol DPNB 50%/IPBC50%,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...
1,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,8.675964e-07,494793-67-8,2,494793-67-8,Penflufen,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...
2,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,2.669894e-08,494793-67-8,3,494793-67-8,Penflufen,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...
3,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,2.669894e-08,29911-28-2,4,29911-28-2,Areosol DPNB,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...
4,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,1.000000e+00,No CAS,5,No CAS,Wood Versowood,Versowood,1,Superwood® fully impregnated\t\t → Treated woo...
5,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,9.999991e-01,...,NaN,NaN,9.999991e-01,No CAS,6,No CAS,Wood Er-Saha Oy,Er-Saha Oy,1,Superwood® fully impregnated\t\t → Treated woo...
6,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,1.000000e+00,No CAS,7,No CAS,Wood Koskisen Oyj,Koskisen Oyj,1,Superwood® fully impregnated\t\t → Treated woo...
7,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,3.000000e-08,...,NaN,NaN,3.000000e-08,No CAS,8,No CAS,Impregnation (SC200)\n,Bodotex A/S,1,Superwood® fully impregnated\t\t → Treated woo...


In [80]:
def identify_alternative_groups(df, tier_level=5):
    df = df.copy()

    def make_group(row, i):
        col_flag = f"Is alternative of tier {i} material"
        col_anchor = f"Is alternative of tier {i} material (of what?)"

        if str(row.get(col_flag, "")).lower() == "yes":
            anchor = row.get(col_anchor)

            if i == 1:
                ref = row.get("Homogenous Material")
            else:
                ref = row.get(f"Tier {i-1} Material")

            return f"T{i}; {row.get('Product')}; {ref}; {anchor}"

        return np.nan

    for i in range(1, tier_level + 1):
        df[f"t{i}_alt_group"] = df.apply(lambda row: make_group(row, i), axis=1)

    return df

In [81]:
df = identify_alternative_groups(df)
df.head()

,Product,Min weight Homogenous material in Product,Max weight Homogenous material in Product,Min % Homogenous material in Product,Max % Homogenous material in Product,Homogenous Material,Min weight Tier 1 material in Homogenous material,Max weight Tier 1 material in Homogenous material,Min % Tier 1 material in Homogenous material,Max % Tier 1 material in Homogenous material,...,CAS,final_material,final_supplier,tier_depth,final_material_map,t1_alt_group,t2_alt_group,t3_alt_group,t4_alt_group,t5_alt_group
0,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,No CAS,Areosol DPNB 50%/IPBC50%,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...,T1; Superwood® fully impregnated\t\t; Treated ...,NaN,NaN,NaN,NaN
1,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,494793-67-8,Penflufen,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...,T1; Superwood® fully impregnated\t\t; Treated ...,NaN,NaN,NaN,NaN
2,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,494793-67-8,Penflufen,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...,T1; Superwood® fully impregnated\t\t; Treated ...,NaN,NaN,NaN,NaN
3,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,29911-28-2,Areosol DPNB,Bodotex A/S,2,Superwood® fully impregnated\t\t → Treated woo...,T1; Superwood® fully impregnated\t\t; Treated ...,NaN,NaN,NaN,NaN
4,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,No CAS,Wood Versowood,Versowood,1,Superwood® fully impregnated\t\t → Treated woo...,T1; Superwood® fully impregnated\t\t; Treated ...,NaN,NaN,NaN,NaN


In [82]:
def generate_scenarios(df, tier_level=5):
    alt_choices = {}

    for i in range(1, tier_level + 1):
        group_col = f"t{i}_alt_group"
        material_col = f"Tier {i} Material"

        subset = df.dropna(subset=[group_col])

        for group, grp in subset.groupby(group_col):
            choices = grp[material_col].dropna().unique().tolist()
            alt_choices[group] = sorted(choices)

    # if no alternatives found
    if not alt_choices:
        return [{"scenario_id": "base", "choices": {}}]

    group_names = list(alt_choices.keys())
    scenarios = []

    for i, combo in enumerate(
        itertools.product(*(alt_choices[g] for g in group_names)),
        start=1
    ):
        choices = dict(zip(group_names, combo))
        scenarios.append({
            "scenario_id": f"scenario_{i}",
            "choices": choices
        })

    return scenarios

In [83]:
scenarios = generate_scenarios(df, tier_level=5)
print(len(scenarios))
print(scenarios)

9
[{'scenario_id': 'scenario_1', 'choices': {'T1; Superwood® fully impregnated\t\t; Treated wood; Impregnation (SC200)\n': 'Impregnation (SC200)\n', 'T1; Superwood® fully impregnated\t\t; Treated wood; Wood Versowood': 'Wood Er-Saha Oy'}}, {'scenario_id': 'scenario_2', 'choices': {'T1; Superwood® fully impregnated\t\t; Treated wood; Impregnation (SC200)\n': 'Impregnation (SC200)\n', 'T1; Superwood® fully impregnated\t\t; Treated wood; Wood Versowood': 'Wood Koskisen Oyj'}}, {'scenario_id': 'scenario_3', 'choices': {'T1; Superwood® fully impregnated\t\t; Treated wood; Impregnation (SC200)\n': 'Impregnation (SC200)\n', 'T1; Superwood® fully impregnated\t\t; Treated wood; Wood Versowood': 'Wood Versowood'}}, {'scenario_id': 'scenario_4', 'choices': {'T1; Superwood® fully impregnated\t\t; Treated wood; Impregnation (SC200)\n': 'Impregnation (SC400) [ALT]', 'T1; Superwood® fully impregnated\t\t; Treated wood; Wood Versowood': 'Wood Er-Saha Oy'}}, {'scenario_id': 'scenario_5', 'choices': {'T

In [84]:
def row_is_active(row, scenario, selected_materials, tier_level=5):
    # make sure if there are substances coupled to an alternative then you find it here:
    for i in range(1, tier_level + 1):
        # Alternative filtering
        alt_group_col = f"t{i}_alt_group"
        material_col = f"Tier {i} Material"

        if pd.notna(row.get(alt_group_col)):
            chosen = scenario["choices"].get(row[alt_group_col])
            if row.get(material_col) != chosen:
                return False, f"Excluded by Tier {i} alternative"

        # Coupling rule
        coupling_col1 = f"Coupled to Tier {i} material (only present if coupled material is present)"
        coupling_col2 = f"Coupled to Tier {i} material (only present if coupled material is present) (of what?)"
        coupled_material = row.get(coupling_col2)
        coupling_yes_no = row.get(coupling_col1)

        if coupling_yes_no == "yes" or coupling_yes_no == "Yes" or coupling_yes_no == "yes " or coupling_yes_no == "Yes ":
            if coupled_material not in selected_materials:
                return False, f"Excluded by Tier {i} coupling"

    return True, "Active"

In [85]:
### Calculate the %
def calc_row_contribution(row, tier_level=3):
    min_val_prod = row["Min % Homogenous material in Product"] * row["Min % Tier 1 material in Homogenous material"]
    max_val_prod = row["Max % Homogenous material in Product"] * row["Max % Tier 1 material in Homogenous material"]

    min_val_hom_mat = row["Min % Tier 1 material in Homogenous material"]
    max_val_hom_mat = row["Max % Tier 1 material in Homogenous material"]
    # Loop over tiers > 1
    for i in range(2, tier_level + 1):
        material_col = f"Tier {i} Material"
        min_col = f"Tier {i} Material Weight% Min"
        max_col = f"Tier {i} Material Weight% Max"

        if pd.notna(row.get(material_col)):
            min_val_prod *= row.get(min_col, 1)
            max_val_prod *= row.get(max_col, 1)
            min_val_hom_mat *= row.get(min_col, 1)
            max_val_hom_mat *= row.get(max_col, 1)


    return min_val_prod, max_val_prod, min_val_hom_mat, max_val_hom_mat

In [86]:
# ### Generate this as a df with scenarios and with the active row
# def evaluate_scenario(df, scenario):
#     df = df.copy()
#     selected_materials = set(scenario["choices"].values())
#     active_flags = []
#     reasons = []
#     min_val_prod_contibutions = []
#     max_val_prod_contibutions = []
#     min_val_hom_mat_contibutions = []
#     max_val_hom_mat_contibutions = []
#     for _, row in df.iterrows():
#         active, reason = row_is_active(row, scenario, selected_materials)
#         active_flags.append(active)
#         reasons.append(reason)
#         if active:
#             min_val_prod, max_val_prod, min_val_hom_mat, max_val_hom_mat = calc_row_contribution(row)
#         else:
#             min_val_prod, max_val_prod, min_val_hom_mat, max_val_hom_mat = np.nan, np.nan, np.nan, np.nan
#
#         min_val_prod_contibutions.append(min_val_prod)
#         max_val_prod_contibutions.append(max_val_prod)
#         min_val_hom_mat_contibutions.append(min_val_hom_mat)
#         max_val_hom_mat_contibutions.append(max_val_hom_mat)
#
#     df["scenario_id"] = scenario["scenario_id"]
#     df["active"] = active_flags
#     df["status_reason"] = reasons
#     df["min_contribution_prod"] = min_val_prod_contibutions
#     df["max_contribution_prod"] = max_val_prod_contibutions
#     df["min_contribution_hom_mat"] = min_val_hom_mat_contibutions
#     df["max_contribution_hom_mat"] = max_val_hom_mat_contibutions
#
#     scenario_df = df.copy()
#     return scenario_df



In [87]:
def evaluate_row_activity(df, scenario):
    df = df.copy()
    selected_materials = set(scenario["choices"].values())
    active_flags = []
    reasons = []

    for _, row in df.iterrows():
        active, reason = row_is_active(row, scenario, selected_materials)
        active_flags.append(active)
        reasons.append(reason)

    df["scenario_id"] = scenario["scenario_id"]
    df["active"] = active_flags
    df["status_reason"] = reasons

    scenario_df = df.copy()
    return scenario_df



In [88]:
def evaluate_row_activity(df, scenario):
    df = df.copy()
    selected_materials = set(scenario["choices"].values())
    active_flags = []
    reasons = []

    for _, row in df.iterrows():
        active, reason = row_is_active(row, scenario, selected_materials)
        active_flags.append(active)
        reasons.append(reason)

    df["scenario_id"] = scenario["scenario_id"]
    df["active"] = active_flags
    df["status_reason"] = reasons

    scenario_df = df.copy()

    return scenario_df

In [89]:
def calculate_material_percentages_product(df):
    df = df.copy()
    df_mass_calc = df.copy()
    keys = ["Product","Min weight Homogenous material in Product",	"Max weight Homogenous material in Product", "Homogenous Material"]
    only_active = df_mass_calc["active"] == True
    df_mass_calc_unique = df.loc[only_active, keys].drop_duplicates()

    def calculations_for_material_percentages_product(df):
        """  Calculate the percentage of material based on mass given (worst & best case scenarios)"""
        df = df.copy()
        min_col = "Min weight Homogenous material in Product"
        max_col = "Max weight Homogenous material in Product"
        group_cols = "Product"
        df["total_min_product"] = df.groupby(group_cols)[min_col].transform("sum")
        df["total_max_product"] = df.groupby(group_cols)[max_col].transform("sum")

        df["rest_min"] = df["total_min_product"] - df[min_col]
        df["rest_max"] = df["total_max_product"] - df[max_col]

        df['Min % Homogenous material in Product'] = df[min_col] / (df[min_col] + df["rest_max"])
        df['Max % Homogenous material in Product'] = df[max_col] / (df[max_col] + df["rest_min"])

        return df
    #calculate_material_percentages_product(df_mass_calc_unique)
    df_mass_calc_unique = calculations_for_material_percentages_product(df_mass_calc_unique)
    #
    df_mass_calc_unique["key"] = list(zip(*(df_mass_calc_unique[k] for k in keys)))
    df["key"] = list(zip(*(df[k] for k in keys)))
    #
    min_map = df_mass_calc_unique.set_index("key")["Min % Homogenous material in Product"]
    max_map = df_mass_calc_unique.set_index("key")["Max % Homogenous material in Product"]

    df["Min % Homogenous material in Product"] = df["Min % Homogenous material in Product"].fillna(df["key"].map(min_map))
    df["Max % Homogenous material in Product"] = df["Max % Homogenous material in Product"].fillna(df["key"].map(max_map))
    df.drop(["key"], axis=1, inplace=True)
    return df

In [90]:
def calculate_material_percentages_hom_mat(df):
    df = df.copy()
    df_mass_calc = df.copy()
    keys = ["Homogenous Material","Min weight Tier 1 material in Homogenous material",	"Max weight Tier 1 material in Homogenous material", "Tier 1 Material"]
    only_active = df_mass_calc["active"] == True
    df_mass_calc_unique = df.loc[only_active, keys].drop_duplicates()

    def calculations_for_material_percentages_hom_mat(df):
        """  Calculate the percentage of material based on mass given (worst & best case scenarios)"""
        df = df.copy()
        min_col = "Min weight Tier 1 material in Homogenous material"
        max_col = "Max weight Tier 1 material in Homogenous material"
        group_cols = "Homogenous Material"
        df["total_min_product"] = df.groupby(group_cols)[min_col].transform("sum")
        df["total_max_product"] = df.groupby(group_cols)[max_col].transform("sum")

        df["rest_min"] = df["total_min_product"] - df[min_col]
        df["rest_max"] = df["total_max_product"] - df[max_col]

        df['Min % Tier 1 material in Homogenous material'] = df[min_col] / (df[min_col] + df["rest_max"])
        df['Max % Tier 1 material in Homogenous material'] = df[max_col] / (df[max_col] + df["rest_min"])

        return df
    #calculate_material_percentages_product(df_mass_calc_unique)
    df_mass_calc_unique = calculations_for_material_percentages_hom_mat(df_mass_calc_unique)

    df_mass_calc_unique["key"] = list(zip(*(df_mass_calc_unique[k] for k in keys)))
    df["key"] = list(zip(*(df[k] for k in keys)))

    min_map = df_mass_calc_unique.set_index("key")["Min % Tier 1 material in Homogenous material"]
    max_map = df_mass_calc_unique.set_index("key")["Max % Tier 1 material in Homogenous material"]

    df["Min % Tier 1 material in Homogenous material"] = df["Min % Tier 1 material in Homogenous material"].fillna(df["key"].map(min_map))
    df["Max % Tier 1 material in Homogenous material"] = df["Max % Tier 1 material in Homogenous material"].fillna(df["key"].map(max_map))
    df.drop(["key"], axis=1, inplace=True)
    return df

In [91]:
# ### calculating the % based on the mass of the homogenous materials in the product and tier 1
# def calculate_percentage_based_on_mass(scenario_df):
#     def calculate_material_percentages_hom_mat(scenario_df):
#         """  Calculate the percentage of material based on mass given (worst & best case scenarios)"""
#         df = scenario_df.copy()
#         min_col = "Min weight Tier 1 material in Homogenous material"
#         max_col = "Max weight Tier 1 material in Homogenous material"
#         group_cols = "Homogenous Material"
#         df["total_min_product"] = df.groupby(group_cols)[min_col].transform("sum")
#         df["total_max_product"] = df.groupby(group_cols)[max_col].transform("sum")
#
#         df["rest_min"] = df["total_min_product"] - df[min_col]
#         df["rest_max"] = df["total_max_product"] - df[max_col]
#
#         df['min_frac_hom'] = df[min_col] / (df[min_col] + df["rest_max"])
#         df['max_frac_hom'] = df[max_col] / (df[max_col] + df["rest_min"])
#
#         return df
#     #calculate_material_percentages_product(df_mass_calc_unique)
#     df_mass_calc_unique = calculate_material_percentages_hom_mat(scenario_df)
#
#     scenario_df = scenario_df.merge(
#         df_mass_calc_unique[
#             [
#                 "Homogenous Material",
#                 "Min weight Tier 1 material in Homogenous material",
#                 "Max weight Tier 1 material in Homogenous material",
#                 "Tier 1 Material",
#                 "min_frac_hom",
#                 "max_frac_hom",
#             ]
#         ],
#         on=[
#             "Homogenous Material",
#             "Min weight Tier 1 material in Homogenous material",
#             "Max weight Tier 1 material in Homogenous material",
#             "Tier 1 Material",
#         ],
#         how="left"
#     )
#     scenario_df["Min % Tier 1 material in Homogenous material"] = scenario_df["min_frac_hom"]
#     scenario_df["Max % Tier 1 material in Homogenous material"] = scenario_df["max_frac_hom"]
#
#     scenario_df = scenario_df.drop(columns=["min_frac_hom", "max_frac_hom"])
#     return scenario_df

In [92]:
### calculating the % in product and hom mat
def calculate_row_contributions(df):
    df = df.copy()

    min_val_prod_contibutions = []
    max_val_prod_contibutions = []
    min_val_hom_mat_contibutions = []
    max_val_hom_mat_contibutions = []
    for _, row in df.iterrows():
        if row.get("active") is True:
            min_val_prod, max_val_prod, min_val_hom_mat, max_val_hom_mat = calc_row_contribution(row)
        else:
            min_val_prod, max_val_prod, min_val_hom_mat, max_val_hom_mat = np.nan, np.nan, np.nan, np.nan

        min_val_prod_contibutions.append(min_val_prod)
        max_val_prod_contibutions.append(max_val_prod)
        min_val_hom_mat_contibutions.append(min_val_hom_mat)
        max_val_hom_mat_contibutions.append(max_val_hom_mat)


    df["min_contribution_prod"] = min_val_prod_contibutions
    df["max_contribution_prod"] = max_val_prod_contibutions
    df["min_contribution_hom_mat"] = min_val_hom_mat_contibutions
    df["max_contribution_hom_mat"] = max_val_hom_mat_contibutions

    calc_df = df.copy()
    return calc_df


In [93]:
### generate all options
def generate_all_options(df, scenarios):
    all_evaluated_scenarios = []
    for scenario in scenarios:
        scenario_df = evaluate_row_activity(df, scenario)
        product_percent_df = calculate_material_percentages_product(scenario_df)
        hom_mat_percent_df = calculate_material_percentages_hom_mat(product_percent_df)
        scenario_evaluated = calculate_row_contributions(hom_mat_percent_df)
        all_evaluated_scenarios.append(scenario_evaluated)

    all_evaluated_scenarios_df = pd.concat(all_evaluated_scenarios, ignore_index=True)
    return all_evaluated_scenarios_df


In [115]:
def update_low(record, key, value, scenario_id):
    if pd.isna(value):
        return
    value_col = f"{key}_value"
    scenario_col = f"{key}_scenario"

    if value_col not in record or pd.isna(record[value_col]) or value < record[value_col]:
        record[value_col] = value
        record[scenario_col] = scenario_id

def update_high(record, key, value, scenario_id):
    if pd.isna(value):
        return
    value_col = f"{key}_value"
    scenario_col = f"{key}_scenario"

    if value_col not in record or pd.isna(record[value_col]) or value > record[value_col]:
        record[value_col] = value
        record[scenario_col] = scenario_id


metrics = [
    "min_contribution_prod",
    "max_contribution_prod",
    "min_contribution_hom_mat",
    "max_contribution_hom_mat"
]

summary = {}
scenario_extremes = {}


for scenario in scenarios:
    scenario_df = evaluate_row_activity(df, scenario)
    product_percent_df = calculate_material_percentages_product(scenario_df)
    hom_mat_percent_df = calculate_material_percentages_hom_mat(product_percent_df)
    scenario_evaluated = calculate_row_contributions(hom_mat_percent_df).copy()

    # Keep only rows that actually have contributions
    active_mask = scenario_evaluated["active"].astype(str).str.upper().eq("TRUE")
    current = scenario_evaluated.loc[
        active_mask,
        ["row_id", "CAS", "final_material", "final_material_map", "scenario_id"] + metrics
    ].copy()

    # Update running absolute bounds
    for row in current.itertuples(index=False):
        row_id = row.row_id
        cas = row.CAS
        material = row.final_material
        material_map = row.final_material_map
        scenario_id = row.scenario_id

        key = (row_id, cas, material, material_map)

        rec = summary.setdefault(
            key,
            {
                "row_id": row_id,
                "CAS": cas,
                "final_material": material,
                "final_material_map": material_map,
            }
        )

        update_low(rec,  "abs_min_contribution_prod",    row.min_contribution_prod,    scenario_id)
        update_high(rec, "abs_max_contribution_prod",    row.max_contribution_prod,    scenario_id)
        update_low(rec,  "abs_min_contribution_hom_mat", row.min_contribution_hom_mat, scenario_id)
        update_high(rec, "abs_max_contribution_hom_mat", row.max_contribution_hom_mat, scenario_id)

    ####
    scenario_summaries = {}
    # Keep only active rows
    active_mask = scenario_evaluated["active"].astype(str).str.upper().eq("TRUE")

    current = scenario_evaluated.loc[
        active_mask,
        ["scenario_id", "CAS", "min_contribution_prod", "max_contribution_prod"]
    ].copy()

    # Convert to numeric (important because of commas)
    for col in ["min_contribution_prod", "max_contribution_prod"]:
        current[col] = pd.to_numeric(
            current[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )

    # Sum per scenario
    scenario_id = current["scenario_id"].iloc[0]
    # Filter out rows where CAS is "No CAS"
    current = current[current["CAS"] != "No CAS"]

    sum_min = current["min_contribution_prod"].sum(skipna=True)
    sum_max = current["max_contribution_prod"].sum(skipna=True)

    rec = scenario_extremes.setdefault("global", {})

    # MIN of min_contribution_prod (worst-case lowest)
    if "abs_min_sum_min_prod" not in rec or sum_min < rec["abs_min_sum_min_prod"]:
        rec["abs_min_sum_min_prod"] = sum_min
        rec["abs_min_sum_min_prod_scenario"] = scenario_id

    # MAX of min_contribution_prod
    if "abs_max_sum_min_prod" not in rec or sum_min > rec["abs_max_sum_min_prod"]:
        rec["abs_max_sum_min_prod"] = sum_min
        rec["abs_max_sum_min_prod_scenario"] = scenario_id

    # MIN of max_contribution_prod
    if "abs_min_sum_max_prod" not in rec or sum_max < rec["abs_min_sum_max_prod"]:
        rec["abs_min_sum_max_prod"] = sum_max
        rec["abs_min_sum_max_prod_scenario"] = scenario_id

    # MAX of max_contribution_prod
    if "abs_max_sum_max_prod" not in rec or sum_max > rec["abs_max_sum_max_prod"]:
        rec["abs_max_sum_max_prod"] = sum_max
        rec["abs_max_sum_max_prod_scenario"] = scenario_id

    scenario_summaries[scenario_id] = {"scenario_id": scenario_id, "sum_min_contribution_prod": sum_min, "sum_max_contribution_prod": sum_max}
    scenario_summary_df = pd.DataFrame(scenario_summaries.values())

summary_df = (pd.DataFrame(summary.values()).sort_values("row_id").reset_index(drop=True))
# best & worst case:

scenario_extremes_df = pd.DataFrame([scenario_extremes["global"]])

In [95]:
summary_df.to_excel("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/summary_test_scenarios_31-03.xlsx", index=False)

In [96]:
scenario_extremes_df.head()

,abs_min_sum_min_prod,abs_min_sum_min_prod_scenario,abs_max_sum_min_prod,abs_max_sum_min_prod_scenario,abs_min_sum_max_prod,abs_min_sum_max_prod_scenario,abs_max_sum_max_prod,abs_max_sum_max_prod_scenario
0,0.999999,scenario_4,0.999999,scenario_3,0.999999,scenario_4,1.0,scenario_2


In [97]:
scenario_extremes_df.to_excel("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/summary_test_scenarios_superwoods_scenario_extremes-31-03.xlsx", index=False)

In [98]:
def build_selected_scenarios_df(df, scenarios, selected_scenario_ids):
    results = []

    selected_set = set(selected_scenario_ids)

    for scenario in scenarios:
        if scenario["scenario_id"] not in selected_set:
            continue

        scenario_df = evaluate_row_activity(df, scenario)
        product_percent_df = calculate_material_percentages_product(scenario_df)
        hom_mat_percent_df = calculate_material_percentages_hom_mat(product_percent_df)
        scenario_evaluated = calculate_row_contributions(hom_mat_percent_df).copy()
        results.append(scenario_evaluated)

    if results:
        return pd.concat(results, ignore_index=True)

    return pd.DataFrame()
selected_df = build_selected_scenarios_df(df, scenarios, ["scenario_1", "scenario_2"])
selected_df.to_excel("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/test_scenario_extraction_superwoods-31-03.xlsx", index=False)

In [99]:
# def build_absolute_summary(df, scenarios):
#     summary = {}
#
#     for scenario in scenarios:
#         scenario_df = evaluate_row_activity(df, scenario)
#         product_percent_df = calculate_material_percentages_product(scenario_df)
#         hom_mat_percent_df = calculate_material_percentages_hom_mat(product_percent_df)
#         scenario_evaluated = calculate_row_contributions(hom_mat_percent_df).copy()
#
#         active_mask = scenario_evaluated["active"].astype(str).str.upper().eq("TRUE")
#         current = scenario_evaluated.loc[
#             active_mask,
#             ["final_material", "scenario_id",
#              "min_contribution_prod", "max_contribution_prod",
#              "min_contribution_hom_mat", "max_contribution_hom_mat"]
#         ].copy()
#
#         for col in [
#             "min_contribution_prod", "max_contribution_prod",
#             "min_contribution_hom_mat", "max_contribution_hom_mat"
#         ]:
#             current[col] = pd.to_numeric(
#                 current[col].astype(str).str.replace(",", ".", regex=False),
#                 errors="coerce"
#             )
#
#         current = current.groupby(["final_material", "scenario_id"], as_index=False).agg({
#             "min_contribution_prod": "min",
#             "max_contribution_prod": "max",
#             "min_contribution_hom_mat": "min",
#             "max_contribution_hom_mat": "max",
#         })
#
#         for row in current.itertuples(index=False):
#             rec = summary.setdefault(row.final_material, {"final_material": row.final_material})
#
#             update_low(rec,  "abs_min_contribution_prod",    row.min_contribution_prod,    row.scenario_id)
#             update_high(rec, "abs_max_contribution_prod",    row.max_contribution_prod,    row.scenario_id)
#             update_low(rec,  "abs_min_contribution_hom_mat", row.min_contribution_hom_mat, row.scenario_id)
#             update_high(rec, "abs_max_contribution_hom_mat", row.max_contribution_hom_mat, row.scenario_id)
#
#     return pd.DataFrame(summary.values())

In [100]:
# summary_df = build_absolute_summary(df, scenarios)
# summary_df.head(20)

In [101]:
# metrics = ["min_contribution_prod", "max_contribution_prod", "min_contribution_hom_mat", "max_contribution_hom_mat"]
# results = {}
# for scenario in scenarios:
#     scenario_df = evaluate_row_activity(df, scenario)
#     product_percent_df = calculate_material_percentages_product(scenario_df)
#     hom_mat_percent_df = calculate_material_percentages_hom_mat(product_percent_df)
#     scenario_evaluated = calculate_row_contributions(hom_mat_percent_df).copy()
#
#     for _, row in scenario_evaluated.iterrows():
#         row_id = row[["row_id"]]
#         min_contribution_prod = row["min_contribution_prod"]
#         max_contribution_prod = row["max_contribution_prod"]
#         min_contribution_hom_mat = row["min_contribution_hom_mat"]
#         max_contribution_hom_mat = row["max_contribution_hom_mat"]
#
#         if row_id not in results:
#             results[row_id] = {"row_id": row_id}
#             print(pd.DataFrame(results.values()))
#
#
# #
# #         if mat not in results:
# #             results[mat] = {"final_material": mat}
# #
# #             for metric in metrics:
# #                 results[mat][f"{metric}_abs_min"] = row[metric]
# #                 results[mat][f"{metric}_abs_min_scenario"] = row["scenario_id"]
# #                 results[mat][f"{metric}_abs_max"] = row[metric]
# #                 results[mat][f"{metric}_abs_max_scenario"] = row["scenario_id"]
# #         else:
# #             for metric in metrics:
# #                 val = row[metric]
# #
# #                 if val < results[mat][f"{metric}_abs_min"]:
# #                     results[mat][f"{metric}_abs_min"] = val
# #                     results[mat][f"{metric}_abs_min_scenario"] = row["scenario_id"]
# #
# #                 if val > results[mat][f"{metric}_abs_max"]:
# #                     results[mat][f"{metric}_abs_max"] = val
# #                     results[mat][f"{metric}_abs_max_scenario"] = row["scenario_id"]
# # pd.DataFrame(results.values())


In [102]:
# def generate_all_options(df, scenarios):
#     all_evaluated_scenarios = []
#     for scenario in scenarios:
#         scenario_evaluated = evaluate_scenario(df, scenario)
#         all_evaluated_scenarios.append(scenario_evaluated)
#
#     all_evaluated_scenarios_df = pd.concat(all_evaluated_scenarios, ignore_index=True)
#     return all_evaluated_scenarios_df

In [103]:
# df_scenarios = generate_all_options(df, scenarios)
# df_scenarios.head(15)
# df_scenarios.to_excel("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/MAS_automation_calculations_superwood.xlsx", index=False)
# df_summary = df_scenarios[['row_id','CAS','final_material','final_supplier','tier_depth','final_material_map','t1_alt_group','t2_alt_group','t3_alt_group','t4_alt_group','t5_alt_group','scenario_id','active','status_reason','min_contribution_prod','max_contribution_prod','min_contribution_hom_mat','max_contribution_hom_mat']]
# df_summary.to_excel("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/summary_test_superwood.xlsx", index=False)

In [116]:
scenario_ids = ["scenario_1", "scenario_2", "scenario_3", "scenario_4"]

In [123]:
def select_scenarios(scenario_ids: list) -> list:
    print("Available Scenarios:")
    for i, scenario in enumerate(scenario_ids, 1):
        print(f"  {i}. {scenario}")

    print("Enter the numbers of the scenarios you want (e.g: 1,3,5) or 'all' to select all or X for no scenarios:")

    while True:
        user_input = input(">>> ").strip().lower()

        if user_input == "all":
            selected = scenario_ids[:]
            break

        if user_input == "x":
            selected = []
            break

        try:
            indices = [int(x.strip()) for x in user_input.split(",")]
            if all(1 <= i <= len(scenario_ids) for i in indices):
                selected = [scenario_ids[i - 1] for i in indices]
                break
            else:
                print(f"Please enter numbers between 1 and {len(scenario_ids)}")
        except ValueError:
            print("Invalid input. Use comma-separated numbers like: 1,3,5")

    #print(f"Selected scenarios: {selected}")
    return selected

chosen = select_scenarios(scenario_ids)

Available Scenarios:
  1. Scenario_A
  2. Scenario_B
  3. Scenario_C
  4. Scenario_D
  5. Scenario_E
  6. Scenario_F
  7. Scenario_G
  8. Scenario_H
Enter the numbers of the scenarios you want (e.g: 1,3,5) or 'all' to select all or X for no scenarios:

✅ Selected scenarios: []


In [1]:
import tkinter as tk

root = tk.Tk()
root.title("My App")         # window title
root.geometry("400x300")     # width x height
root.resizable(False, False) # lock width, height
root.configure(bg="#1e1e2e") # background color

def my_function():
    print("Hello World!")

tk.Label(root, text="Hello", font=("Helvetica", 14), fg="white", bg="black").pack()
tk.Button(root, text="Click Me", command=my_function).pack()
tk.Entry(root, width=30).pack()
root.mainloop()

Hello World!
Hello World!
Hello World!


In [6]:
import customtkinter as ctk

ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

root = ctk.CTk()
root.geometry("400x300")

ctk.CTkLabel(root, text="Hello!", font=("Helvetica", 20)).pack(pady=20)
ctk.CTkButton(root, text="Click Me", command=lambda: print("clicked")).pack()
ctk.CTkEntry(root, placeholder_text="Type here...").pack(pady=10)

root.mainloop()

clicked
clicked
clicked


In [119]:
# import tkinter as tk
# from tkinter import messagebox
#
#
# def select_scenarios(scenario_ids: list) -> list:
#     selected = []
#
#     # ── Window setup ──────────────────────────────────────────────
#     root = tk.Tk()
#     root.title("Scenario Selector")
#     root.resizable(False, False)
#     root.attributes("-topmost", True)
#
#     BG       = "#1e1e2e"
#     SURFACE  = "#2a2a3e"
#     ACCENT   = "#7c6af7"
#     ACCENT_H = "#9d8fff"
#     FG       = "#e0e0f0"
#     FG_DIM   = "#888aaa"
#     CHECK_ON = "#7c6af7"
#
#     root.configure(bg=BG)
#
#     # ── Header ────────────────────────────────────────────────────
#     header = tk.Frame(root, bg=BG)
#     header.pack(fill="x", padx=24, pady=(20, 0))
#
#     tk.Label(header, text="Select Scenarios", font=("Helvetica", 16, "bold"),
#              bg=BG, fg=FG).pack(anchor="w")
#     tk.Label(header, text="Choose one or more scenarios to include in your run.",
#              font=("Helvetica", 10), bg=BG, fg=FG_DIM).pack(anchor="w", pady=(2, 0))
#
#     # ── Separator ─────────────────────────────────────────────────
#     tk.Frame(root, bg=ACCENT, height=1).pack(fill="x", padx=24, pady=12)
#
#     # ── Scrollable checklist ──────────────────────────────────────
#     container = tk.Frame(root, bg=BG)
#     container.pack(fill="both", expand=True, padx=24)
#
#     canvas = tk.Canvas(container, bg=BG, highlightthickness=0,
#                        width=340, height=min(len(scenario_ids) * 42, 280))
#     scrollbar = tk.Scrollbar(container, orient="vertical", command=canvas.yview)
#     canvas.configure(yscrollcommand=scrollbar.set)
#
#     if len(scenario_ids) > 6:
#         scrollbar.pack(side="right", fill="y")
#     canvas.pack(side="left", fill="both", expand=True)
#
#     inner = tk.Frame(canvas, bg=BG)
#     canvas_window = canvas.create_window((0, 0), window=inner, anchor="nw")
#
#     def on_frame_configure(e):
#         canvas.configure(scrollregion=canvas.bbox("all"))
#
#     def on_canvas_configure(e):
#         canvas.itemconfig(canvas_window, width=e.width)
#
#     inner.bind("<Configure>", on_frame_configure)
#     canvas.bind("<Configure>", on_canvas_configure)
#     canvas.bind_all("<MouseWheel>", lambda e: canvas.yview_scroll(-1 * (e.delta // 120), "units"))
#
#     # ── Checkboxes ────────────────────────────────────────────────
#     vars_ = []
#     for scenario in scenario_ids:
#         var = tk.BooleanVar()
#         vars_.append(var)
#
#         row = tk.Frame(inner, bg=SURFACE, cursor="hand2")
#         row.pack(fill="x", pady=3, ipady=6, ipadx=10)
#
#         cb = tk.Checkbutton(
#             row, text=f"  {scenario}", variable=var,
#             font=("Helvetica", 11), bg=SURFACE, fg=FG,
#             selectcolor=CHECK_ON, activebackground=SURFACE,
#             activeforeground=FG, relief="flat", anchor="w",
#             highlightthickness=0
#         )
#         cb.pack(fill="x")
#
#         # Hover effect
#         row.bind("<Enter>", lambda e, f=row: f.configure(bg="#33334d"))
#         row.bind("<Leave>", lambda e, f=row: f.configure(bg=SURFACE))
#         cb.bind("<Enter>",  lambda e, f=row: f.configure(bg="#33334d"))
#         cb.bind("<Leave>",  lambda e, f=row: f.configure(bg=SURFACE))
#
#     # ── Select All toggle ─────────────────────────────────────────
#     tk.Frame(root, bg="#3a3a5c", height=1).pack(fill="x", padx=24, pady=(10, 0))
#
#     toggle_frame = tk.Frame(root, bg=BG)
#     toggle_frame.pack(fill="x", padx=24, pady=6)
#
#     def toggle_all():
#         state = not all(v.get() for v in vars_)
#         for v in vars_:
#             v.set(state)
#         all_btn.configure(text="☑  Deselect All" if state else "☐  Select All")
#
#     all_btn = tk.Button(
#         toggle_frame, text="☐  Select All",
#         font=("Helvetica", 10), bg=BG, fg=ACCENT,
#         relief="flat", cursor="hand2", activeforeground=ACCENT_H,
#         activebackground=BG, command=toggle_all
#     )
#     all_btn.pack(anchor="w")
#
#     # ── Confirm button ────────────────────────────────────────────
#     def confirm():
#         chosen = [scenario_ids[i] for i, v in enumerate(vars_) if v.get()]
#         if not chosen:
#             messagebox.showwarning("No Selection", "Please select at least one scenario.")
#             return
#         selected.extend(chosen)
#         root.destroy()
#
#     btn_frame = tk.Frame(root, bg=BG)
#     btn_frame.pack(fill="x", padx=24, pady=(6, 20))
#
#     confirm_btn = tk.Button(
#         btn_frame, text="Confirm Selection",
#         font=("Helvetica", 11, "bold"),
#         bg=ACCENT, fg="white", relief="flat",
#         cursor="hand2", activebackground=ACCENT_H,
#         activeforeground="white", padx=20, pady=8,
#         command=confirm
#     )
#     confirm_btn.pack(fill="x")
#
#     # ── Center window on screen ───────────────────────────────────
#     root.update_idletasks()
#     w, h = root.winfo_width(), root.winfo_height()
#     x = (root.winfo_screenwidth()  - w) // 2
#     y = (root.winfo_screenheight() - h) // 2
#     root.geometry(f"+{x}+{y}")
#
#     root.mainloop()
#
#     print(f"✅ Selected scenarios: {selected}")
#     return selected
#
#
# # ── Example usage ─────────────────────────────────────────────────
# if __name__ == "__main__":
#     scenario_ids = [
#         "Scenario_A", "Scenario_B", "Scenario_C",
#         "Scenario_D", "Scenario_E", "Scenario_F",
#         "Scenario_G", "Scenario_H",
#     ]
#
#     chosen = select_scenarios(scenario_ids)

✅ Selected scenarios: ['Scenario_B', 'Scenario_D', 'Scenario_G']


In [104]:
df_scenarios = generate_all_options(df, scenarios)
df_scenarios.head(15)

# df_scenarios.to_csv("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/MAS_automation_calculations_superwood.csv", index=False)
# df_summary = df_scenarios[['row_id','CAS','final_material','final_supplier','tier_depth','final_material_map','t1_alt_group','t2_alt_group','t3_alt_group','t4_alt_group','t5_alt_group','scenario_id','active','status_reason','min_contribution_prod','max_contribution_prod','min_contribution_hom_mat','max_contribution_hom_mat']]
# df_summary.to_csv("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/summary_test_superwood.csv", index=False)

,Product,Min weight Homogenous material in Product,Max weight Homogenous material in Product,Min % Homogenous material in Product,Max % Homogenous material in Product,Homogenous Material,Min weight Tier 1 material in Homogenous material,Max weight Tier 1 material in Homogenous material,Min % Tier 1 material in Homogenous material,Max % Tier 1 material in Homogenous material,...,t3_alt_group,t4_alt_group,t5_alt_group,scenario_id,active,status_reason,min_contribution_prod,max_contribution_prod,min_contribution_hom_mat,max_contribution_hom_mat
0,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,scenario_1,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN
1,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,scenario_1,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN
2,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,scenario_1,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN
3,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,2.669894e-08,...,NaN,NaN,NaN,scenario_1,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN
4,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,NaN,scenario_1,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN
5,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,9.999991e-01,...,NaN,NaN,NaN,scenario_1,True,Active,9.999993e-01,9.999991e-01,9.999993e-01,9.999991e-01
6,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,449455.950,461043.450,9.999993e-01,1.000000e+00,...,NaN,NaN,NaN,scenario_1,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN
7,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.012,0.012,2.602791e-08,3.000000e-08,...,NaN,NaN,NaN,scenario_1,True,Active,2.602791e-08,3.000000e-08,2.602791e-08,3.000000e-08
8,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,scenario_2,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN
9,Superwood® fully impregnated\t\t,NaN,NaN,1,1,Treated wood,0.300,0.400,6.506975e-07,8.675964e-07,...,NaN,NaN,NaN,scenario_2,False,Excluded by Tier 1 alternative,NaN,NaN,NaN,NaN


In [105]:
df_scenarios.shape

(72, 76)

In [106]:
# df_scenarios.to_csv("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/MAS_automation_calculations_superwood.csv", index=False)

In [107]:
df_scenarios.to_excel("/Users/juliakulpa/Library/CloudStorage/Dropbox-Arche/Julia Kulpa/MAS_automation_calculations_superwood-31-03.xlsx", index=False)

# when we have the per scenario then first check: if only percentages are present for all, if not then calculate the precentage based on the mass
# what if this is not fully ? how to then calculate ?
# then calculate the percentages: for homogenous material and for the whole product
# for all scenarios do min and max values based on all scenarios

In [108]:
# df_mass_calc = df_scenarios.copy()
#
# df_mass_calc = df_mass_calc[df_mass_calc["scenario_id"]=="scenario_1"]
# df = df_mass_calc.copy()
# only_active = df_mass_calc["active"] == True
# df_mass_calc_unique = df.loc[only_active, ["Homogenous Material","Min weight Tier 1 material in Homogenous material",	"Max weight Tier 1 material in Homogenous material", "Tier 1 Material"]].drop_duplicates()
# df_mass_calc_unique.head(15)
# def calculate_material_percentages_product(df):
#     """  Calculate the percentage of material based on mass given (worst & best case scenarios)
#     """
#     df = df.copy()
#
#     min_col = "Min weight Homogenous material in Product"
#     max_col = "Max weight Homogenous material in Product"
#     min_frac = "Min % Homogenous material in Product"
#     max_frac = "Max % Homogenous material in Product"
#     group_cols = ["Product"]
#     df["total_min_product"] = df.groupby(group_cols)[min_col].transform("sum")
#     df["total_max_product"] = df.groupby(group_cols)[max_col].transform("sum")
#
#     df["rest_min"] = df["total_min_product"] - df[min_col]
#     df["rest_max"] = df["total_max_product"] - df[max_col]
#
#     df[min_frac] = df[min_col] / (df[min_col] + df["rest_max"])
#     df[max_frac] = df[max_col] / (df[max_col] + df["rest_min"])
#
#     return df
# def calculate_material_percentages_hom_mat(df):
#     """  Calculate the percentage of material based on mass given (worst & best case scenarios)"""
#     df = df.copy()
#     min_col = "Min weight Tier 1 material in Homogenous material"
#     max_col = "Max weight Tier 1 material in Homogenous material"
#     group_cols = "Homogenous Material"
#     df["total_min_product"] = df.groupby(group_cols)[min_col].transform("sum")
#     df["total_max_product"] = df.groupby(group_cols)[max_col].transform("sum")
#
#     df["rest_min"] = df["total_min_product"] - df[min_col]
#     df["rest_max"] = df["total_max_product"] - df[max_col]
#
#     df['Min % Tier 1 material in Homogenous material'] = df[min_col] / (df[min_col] + df["rest_max"])
#     df['Max % Tier 1 material in Homogenous material'] = df[max_col] / (df[max_col] + df["rest_min"])
#
#     return df
# #calculate_material_percentages_product(df_mass_calc_unique)
# df_mass_calc_unique = calculate_material_percentages_hom_mat(df_mass_calc_unique)
#
# keys = ["Homogenous Material","Min weight Tier 1 material in Homogenous material",	"Max weight Tier 1 material in Homogenous material", "Tier 1 Material"]
#
# df_mass_calc_unique["key"] = list(zip(*(df_mass_calc_unique[k] for k in keys)))
# df["key"] = list(zip(*(df[k] for k in keys)))
#
# min_map = df_mass_calc_unique.set_index("key")["Min % Tier 1 material in Homogenous material"]
# max_map = df_mass_calc_unique.set_index("key")["Max % Tier 1 material in Homogenous material"]
#
# df["Min % Tier 1 material in Homogenous material"] = df["Min % Tier 1 material in Homogenous material"].fillna(df["key"].map(min_map))
# df["Max % Tier 1 material in Homogenous material"] = df["Max % Tier 1 material in Homogenous material"].fillna(df["key"].map(max_map))
# df.drop(["key"], axis=1, inplace=True)
#
# df.head(15)
# #
# # df = df.merge(
# #     df_mass_calc_unique[
# #         [
# #             "Homogenous Material",
# #             "Min weight Tier 1 material in Homogenous material",
# #             "Max weight Tier 1 material in Homogenous material",
# #             "Tier 1 Material",
# #             "min_frac_hom",
# #             "max_frac_hom",
# #         ]
# #     ],
# #     on=[
# #         "Homogenous Material",
# #         "Min weight Tier 1 material in Homogenous material",
# #         "Max weight Tier 1 material in Homogenous material",
# #         "Tier 1 Material",
# #     ],
# #     how="left"
# # )
# # df["Min % Tier 1 material in Homogenous material"] = df["min_frac_hom"]
# # df["Max % Tier 1 material in Homogenous material"] = df["max_frac_hom"]
# #
# # df = df.drop(columns=["min_frac_hom", "max_frac_hom"])
# # df.head(30)
# # min_sum = df_mass_calc_unique["Min weight Homogenous material in Product"].sum()
# # max_sum = df_mass_calc_unique["Max weight Homogenous material in Product"].sum()
# # print(f"Min sum: {min_sum}")
# # print(f"Max sum: {max_sum}")
# # only_active.head(25)
# # for scenario_1 in df_mass_calc["scenario_id"]:
# #     print
# # df_test = df_mass_calc.groupby("Product")["Min weight Homogenous material in Product"].transform("sum")
# # print(df_test)
# # product_min_weight = df.groupby("Product")["Min weight Homogenous material in Product"].transform("sum")
# # product_max_weight = df.groupby("Product")["Max weight Homogenous material in Product"].transform("sum")

In [109]:
# df_mass_calc = df_scenarios.copy()
#
# df_mass_calc = df_mass_calc[df_mass_calc["scenario_id"]=="scenario_1"]
# df = df_mass_calc.copy()
# def calculate_material_percentages_hom_mat(df):
#     df = df.copy()
#     df_mass_calc = df.copy()
#     keys = ["Homogenous Material","Min weight Tier 1 material in Homogenous material",	"Max weight Tier 1 material in Homogenous material", "Tier 1 Material"]
#     only_active = df_mass_calc["active"] == True
#     df_mass_calc_unique = df.loc[only_active, keys].drop_duplicates()
#
#     def calculations_for_material_percentages_hom_mat(df):
#         """  Calculate the percentage of material based on mass given (worst & best case scenarios)"""
#         df = df.copy()
#         min_col = "Min weight Tier 1 material in Homogenous material"
#         max_col = "Max weight Tier 1 material in Homogenous material"
#         group_cols = "Homogenous Material"
#         df["total_min_product"] = df.groupby(group_cols)[min_col].transform("sum")
#         df["total_max_product"] = df.groupby(group_cols)[max_col].transform("sum")
#
#         df["rest_min"] = df["total_min_product"] - df[min_col]
#         df["rest_max"] = df["total_max_product"] - df[max_col]
#
#         df['Min % Tier 1 material in Homogenous material'] = df[min_col] / (df[min_col] + df["rest_max"])
#         df['Max % Tier 1 material in Homogenous material'] = df[max_col] / (df[max_col] + df["rest_min"])
#
#         return df
#     #calculate_material_percentages_product(df_mass_calc_unique)
#     df_mass_calc_unique = calculations_for_material_percentages_hom_mat(df_mass_calc_unique)
#
#     df_mass_calc_unique["key"] = list(zip(*(df_mass_calc_unique[k] for k in keys)))
#     df["key"] = list(zip(*(df[k] for k in keys)))
#
#     min_map = df_mass_calc_unique.set_index("key")["Min % Tier 1 material in Homogenous material"]
#     max_map = df_mass_calc_unique.set_index("key")["Max % Tier 1 material in Homogenous material"]
#
#     df["Min % Tier 1 material in Homogenous material"] = df["Min % Tier 1 material in Homogenous material"].fillna(df["key"].map(min_map))
#     df["Max % Tier 1 material in Homogenous material"] = df["Max % Tier 1 material in Homogenous material"].fillna(df["key"].map(max_map))
#     df.drop(["key"], axis=1, inplace=True)
#     return df
#
# calculate_material_percentages_hom_mat(df)

In [110]:
# df_mass_calc = df_scenarios.copy()
#
# df_mass_calc = df_mass_calc[df_mass_calc["scenario_id"]=="scenario_1"]
# df = df_mass_calc.copy()
# def calculate_material_percentages_product(df):
#     df = df.copy()
#     df_mass_calc = df.copy()
#     keys = ["Product","Min weight Homogenous material in Product",	"Max weight Homogenous material in Product", "Homogenous Material"]
#     only_active = df_mass_calc["active"] == True
#     df_mass_calc_unique = df.loc[only_active, keys].drop_duplicates()
#
#     def calculations_for_material_percentages_product(df):
#         """  Calculate the percentage of material based on mass given (worst & best case scenarios)"""
#         df = df.copy()
#         min_col = "Min weight Homogenous material in Product"
#         max_col = "Max weight Homogenous material in Product"
#         group_cols = "Product"
#         df["total_min_product"] = df.groupby(group_cols)[min_col].transform("sum")
#         df["total_max_product"] = df.groupby(group_cols)[max_col].transform("sum")
#
#         df["rest_min"] = df["total_min_product"] - df[min_col]
#         df["rest_max"] = df["total_max_product"] - df[max_col]
#
#         df['Min % Homogenous material in Product'] = df[min_col] / (df[min_col] + df["rest_max"])
#         df['Max % Homogenous material in Product'] = df[max_col] / (df[max_col] + df["rest_min"])
#
#         return df
#     #calculate_material_percentages_product(df_mass_calc_unique)
#     df_mass_calc_unique = calculations_for_material_percentages_product(df_mass_calc_unique)
#     #
#     df_mass_calc_unique["key"] = list(zip(*(df_mass_calc_unique[k] for k in keys)))
#     df["key"] = list(zip(*(df[k] for k in keys)))
#     #
#     min_map = df_mass_calc_unique.set_index("key")["Min % Homogenous material in Product"]
#     max_map = df_mass_calc_unique.set_index("key")["Max % Homogenous material in Product"]
#
#     df["Min % Homogenous material in Product"] = df["Min % Homogenous material in Product"].fillna(df["key"].map(min_map))
#     df["Max % Homogenous material in Product"] = df["Max % Homogenous material in Product"].fillna(df["key"].map(max_map))
#     df.drop(["key"], axis=1, inplace=True)
#     return df
#
# calculate_material_percentages_product(df)

In [111]:
# # Drop columns that are completely NaN
# for i in range(1, tier_level + 1):
#     col_name_alternatives = f"t{i}_alt_group"
#     if df[col_name_alternatives].isna().all():  # If all values in this column are NaN
#         df = df.drop(columns=[col_name_alternatives])

# # get the highest tier number that has alternatives:
# alt_group_columns = [col for col in df.columns if col.startswith('t') and '_alt_group' in col]
# i_values = [int(col.split('_')[0][1:]) for col in alt_group_columns]
# max_i = max(i_values) if i_values else None

In [ ]:
def get_highest_tier(df, col_pattern):
    numbers = []

    # Convert pattern into regex
    regex_pattern = col_pattern.replace("{i}", r"(\d+)")
    regex_pattern = f"^{regex_pattern}$"

    for col in df.columns:
        match = re.match(regex_pattern, col)
        if match:
            numbers.append(int(match.group(1)))

    if numbers:
        return max(numbers)
    else:
        print("Not determined max tier, set it to 10")
        return 10